# 실습 · 약물 복약상담 RAG를 LangSmith로 관찰하기

이 노트북의 목적은 **LangSmith로 RAG를 추적·평가하는 것을 직접 보는 것**입니다.

복약상담 RAG(약물 정보를 검색해 답하는 챗봇)는 **완성된 코드로 전부 제공**됩니다. 코드를 채울 필요 없이, **위에서부터 순서대로 실행하면서 LangSmith 대시보드에서 무엇이 기록되는지 확인**하면 됩니다.

## 진행 순서

| 단계 | 하는 일 | LangSmith에서 볼 것 |
| --- | --- | --- |
| STEP A~C | 복약상담 RAG 만들기 | (아직 없음) |
| STEP 1 | 추적 켜기 (환경변수 2줄) | - |
| STEP 2 | RAG 실행 | 트레이스 목록 |
| STEP 3 | @traceable 로 단계 추적 | 단계 트리 (RAG→검색→답변) |
| STEP 4 | 트레이스 열어보기 | 프롬프트 전문, 토큰, 시간 |
| STEP 5 | 데이터셋 + 평가 | 점수 표 |
| STEP 6 | v2 만들어 비교 | v1 vs v2 비교 그래프 |

## 준비물

- OpenAI API 키
- LangSmith API 키 (둘 다 무료 발급 — STEP 1에서 안내)

> 각 단계마다 "**대시보드에서 확인**" 안내가 있습니다. 코드 실행 후 https://smith.langchain.com 을 함께 보세요.


---
# 1부. 복약상담 RAG (그대로 실행)

먼저 약물 정보를 검색해 답하는 RAG를 만듭니다. 이 부분은 완성돼 있으니 순서대로 실행만 하세요.


## STEP A. 라이브러리 설치

RAG와 LangSmith에 필요한 패키지를 설치합니다.


## STEP B. 약물 정보 데이터 준비

복약상담에 쓸 약물 정보 8건입니다. 각 약물은 효능, 주의사항, **함께 먹으면 안 되는 약(상호작용)** 정보를 가집니다.

> 학습용 예시 데이터입니다. 실제 복약 판단은 반드시 의사·약사와 상의해야 합니다.


In [1]:
# 약물 정보 (학습용 예시 데이터)
drugs = [
    {"name": "와파린",    "효능": "혈액 응고 방지(항응고제)",
     "주의": "출혈 위험, 정기적 혈액검사 필요",
     "상호작용": "아스피린, 이부프로펜과 함께 복용 시 출혈 위험 증가"},
    {"name": "아스피린",  "효능": "통증 완화, 혈전 예방",
     "주의": "위장 출혈 가능, 공복 복용 피할 것",
     "상호작용": "와파린과 병용 시 출혈 위험, 이부프로펜과 효과 상쇄"},
    {"name": "메트포르민","효능": "제2형 당뇨 혈당 조절",
     "주의": "신장 기능 저하 시 주의, 조영제 검사 전 중단",
     "상호작용": "과도한 음주 시 젖산산증 위험"},
    {"name": "암로디핀",  "효능": "고혈압, 협심증 치료(혈압강하제)",
     "주의": "발목 부종, 어지럼증 가능",
     "상호작용": "자몽주스와 함께 복용 시 혈중 농도 상승"},
    {"name": "심바스타틴","효능": "콜레스테롤 저하(스타틴)",
     "주의": "근육통 시 즉시 상담, 야간 복용 권장",
     "상호작용": "자몽주스, 일부 항생제와 병용 시 근육손상 위험"},
    {"name": "오메프라졸","효능": "위산 분비 억제(위염, 역류성식도염)",
     "주의": "장기 복용 시 골절 위험, 식전 복용",
     "상호작용": "클로피도그렐 효과 감소시킬 수 있음"},
    {"name": "레보티록신","효능": "갑상선 기능 저하증 호르몬 보충",
     "주의": "공복 복용, 복용 후 30분 음식 피할 것",
     "상호작용": "칼슘, 철분제와 동시 복용 시 흡수 저하"},
    {"name": "이부프로펜","효능": "소염진통(해열, 통증, 염증)",
     "주의": "위장장애, 신장 부담, 식후 복용",
     "상호작용": "와파린, 아스피린과 병용 시 출혈 위험"},
]

print("약물 데이터 준비 완료:", len(drugs), "건")
print("예시:", drugs[0]["name"], "-", drugs[0]["효능"])


약물 데이터 준비 완료: 8 건
예시: 와파린 - 혈액 응고 방지(항응고제)


## STEP C. RAG 만들기 (검색 + 답변)

약물 정보를 FAISS에 넣어 검색하고, 검색 결과로 GPT가 답하는 RAG입니다.

먼저 OpenAI 키를 넣습니다. (sk-... 로 시작)


In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()



True

약물 정보를 검색용 문장으로 합쳐 FAISS에 넣습니다.

In [3]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

docs = []
for d in drugs:
    text = f"{d['name']} 효능:{d['효능']} 주의:{d['주의']} 상호작용:{d['상호작용']}"
    docs.append(Document(page_content=text, metadata={"name": d["name"]}))

vs = FAISS.from_documents(docs, embeddings)
print("FAISS 인덱싱 완료:", len(docs), "건")


/tmp/ipykernel_7907/2102544135.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


FAISS 인덱싱 완료: 8 건


RAG의 핵심 함수 3개입니다. 검색(`retrieve`), 답변(`generate`), 둘을 묶는(`rag`) 구조입니다. 지금은 LangSmith 추적이 없는 순수 RAG입니다.

In [4]:
# ── 검색: 질문과 가까운 약물 정보를 찾는다 ──
def retrieve(question: str):
    results = vs.similarity_search_with_score(question, k=2)
    THRESHOLD = 1.5
    if not results or results[0][1] > THRESHOLD:
        return {"context": "", "found": False}
    return {"context": "\n".join(d.page_content for d, s in results), "found": True}


# ── 답변: 찾은 정보로 GPT 가 답한다 ──
def generate(question: str, context: str):
    prompt = (
        "당신은 복약상담 도우미입니다. 아래 약물 정보만 근거로 한국어로 답하세요.\n"
        "정보에 없는 내용은 지어내지 말고, 약물 복용은 의사·약사와 상의하라고 안내하세요.\n\n"
        f"[약물 정보]\n{context}\n\n[질문] {question}\n[답변]"
    )
    return llm.invoke(prompt).content


# ── RAG: 검색 → (찾으면) 답변, (못 찾으면) 안내 ──
def rag(question: str):
    r = retrieve(question)
    if not r["found"]:
        return "관련 약물 정보를 찾지 못했습니다. 약 이름이나 증상을 다시 입력해 주세요."
    return generate(question, r["context"])

print("RAG 함수 준비 완료")


RAG 함수 준비 완료


RAG가 잘 도는지 확인합니다. (LangSmith 없이도 동작합니다.)

In [5]:
for q in ["와파린이랑 아스피린 같이 먹어도 돼?", "암로디핀 먹을 때 주의할 점은?", "감기약 추천해줘"]:
    print(f"Q: {q}")
    print(f"A: {rag(q)}\n")


Q: 와파린이랑 아스피린 같이 먹어도 돼?
A: 와파린과 아스피린을 함께 복용하는 것은 출혈 위험이 증가할 수 있으므로 주의가 필요합니다. 두 약물의 병용은 의사나 약사와 상의한 후 결정하는 것이 좋습니다. 복용 전 반드시 전문가와 상담하시기 바랍니다.

Q: 암로디핀 먹을 때 주의할 점은?
A: 암로디핀을 복용할 때 주의할 점은 발목 부종과 어지럼증이 발생할 수 있다는 것입니다. 또한, 자몽주스와 함께 복용할 경우 혈중 농도가 상승할 수 있으므로 주의해야 합니다. 약물 복용에 대한 자세한 사항은 반드시 의사나 약사와 상의하시기 바랍니다.

Q: 감기약 추천해줘
A: 감기약에 대한 정보는 제공된 약물 정보에 포함되어 있지 않습니다. 감기약 복용에 대해서는 반드시 의사나 약사와 상담하시기 바랍니다.



세 번째 질문(감기약 추천)은 데이터에 없으니 "찾지 못했습니다"가 나오면 정상입니다. 여기까지가 RAG입니다. 이제 LangSmith를 붙입니다.

---
# 2부. LangSmith 붙이기 (실행하며 대시보드 확인)

지금부터는 LangSmith로 이 RAG를 관찰합니다. 각 셀을 실행한 뒤 대시보드를 함께 보세요.


## STEP 1. 추적 켜기

### LangSmith 가입 & 키 발급

1. https://smith.langchain.com 에 접속해 가입합니다 (무료).
2. **Settings -> API Keys -> Create API Key** 를 누릅니다.
3. 기본값(Personal Access Token, Never) 그대로 두고 **Create API Key**.
4. `lsv2_...` 로 시작하는 키가 나옵니다. 창을 닫으면 다시 못 보니 바로 복사하세요.

### 추적 켜기 — 환경변수 2줄

LangSmith 추적은 환경변수 두 개만 켜면 시작됩니다.


In [6]:
import os

# 추적 스위치 ON
os.environ["LANGSMITH_TRACING"] = "true"

# 발급받은 LangSmith 키


# 트레이스를 모아둘 프로젝트 이름
os.environ["LANGSMITH_PROJECT"] = "복약상담-RAG-실습"

print("추적:", os.environ["LANGSMITH_TRACING"])
print("프로젝트:", os.environ["LANGSMITH_PROJECT"])


추적: true
프로젝트: 복약상담-RAG-실습


`추적: true` 가 나오면 켜진 것입니다.

## STEP 2. RAG 실행 → 트레이스 자동 기록

추적을 켰으니, 이제 RAG를 부르면 **코드를 안 바꿔도 자동으로 기록**됩니다.


In [7]:
for q in ["와파린이랑 아스피린 같이 먹어도 돼?",
          "심바스타틴 먹을 때 자몽 먹어도 되나요?",
          "우주선 연료는 뭐야?"]:
    print(f"Q: {q}")
    print(f"A: {rag(q)}\n")

print("실행 완료!")


Q: 와파린이랑 아스피린 같이 먹어도 돼?
A: 와파린과 아스피린을 함께 복용하는 것은 출혈 위험이 증가할 수 있으므로 주의가 필요합니다. 두 약물의 병용은 의사나 약사와 반드시 상의한 후 결정하는 것이 좋습니다. 복용 전 전문가의 상담을 받으시기 바랍니다.

Q: 심바스타틴 먹을 때 자몽 먹어도 되나요?
A: 심바스타틴을 복용할 때 자몽을 먹는 것은 권장되지 않습니다. 자몽주스와 함께 복용할 경우 근육 손상 위험이 증가할 수 있습니다. 약물 복용에 대한 자세한 사항은 반드시 의사나 약사와 상담하시기 바랍니다.

Q: 우주선 연료는 뭐야?
A: 관련 약물 정보를 찾지 못했습니다. 약 이름이나 증상을 다시 입력해 주세요.

실행 완료!


### 대시보드에서 확인

1. https://smith.langchain.com 접속
2. 왼쪽 **Tracing Projects -> 복약상담-RAG-실습** 프로젝트 열기
3. 방금 던진 질문 3개가 트레이스로 쌓였는지 확인

**볼 것**: 각 트레이스의 **Latency(소요시간)**, **Token(토큰 수)**, **Cost(비용)**. 세 번째 질문("우주선 연료")은 검색에서 걸러져 답변을 안 만들었으니 토큰이 0일 것입니다.

> 추적은 자동으로 켜지지만, 우리가 만든 `retrieve` 같은 일반 함수는 아직 단계로 안 보입니다. 그걸 보이게 하는 게 다음 STEP입니다.


## STEP 3. @traceable 로 단계 추적

`retrieve` / `generate` / `rag` 함수 위에 **`@traceable` 데코레이터**를 붙이면, 대시보드에 `복약RAG -> 검색 -> 답변생성` **트리**가 생깁니다.


In [8]:
from langsmith import traceable

@traceable(name="검색")
def retrieve(question: str):
    results = vs.similarity_search_with_score(question, k=2)
    THRESHOLD = 1.5
    if not results or results[0][1] > THRESHOLD:
        return {"context": "", "found": False}
    return {"context": "\n".join(d.page_content for d, s in results), "found": True}


@traceable(name="답변생성")
def generate(question: str, context: str):
    prompt = (
        "당신은 복약상담 도우미입니다. 아래 약물 정보만 근거로 한국어로 답하세요.\n"
        "정보에 없는 내용은 지어내지 말고, 약물 복용은 의사·약사와 상의하라고 안내하세요.\n\n"
        f"[약물 정보]\n{context}\n\n[질문] {question}\n[답변]"
    )
    return llm.invoke(prompt).content


@traceable(name="복약RAG")
def rag(question: str):
    r = retrieve(question)
    if not r["found"]:
        return "관련 약물 정보를 찾지 못했습니다. 약 이름이나 증상을 다시 입력해 주세요."
    return generate(question, r["context"])

print("@traceable 적용 완료")


@traceable 적용 완료


In [9]:
# 추적되는 함수로 다시 실행
rag("와파린이랑 이부프로펜 같이 먹으면?")
print("실행 완료!")


실행 완료!


### 대시보드에서 확인

대시보드에서 방금 만든 `복약RAG` 트레이스를 열면 트리가 보입니다.

```
복약RAG           (부모)
├─ 검색            (자식 - FAISS 검색)
└─ 답변생성        (자식 - GPT 답변)
   └─ ChatOpenAI   (답변생성 안의 GPT 호출이 자동으로 잡힘)
```

`복약RAG`를 클릭하면 그 안의 `검색`, `답변생성`이 펼쳐집니다.


## STEP 4. 트레이스 열어서 프롬프트 디버깅

이번엔 코드가 아니라 **대시보드를 읽는 단계**입니다. RAG 디버깅에서 가장 중요한 건 "GPT가 실제로 어떤 프롬프트를 받았나"를 보는 것입니다.

### 대시보드에서 확인

1. `복약RAG` 트레이스를 연다.
2. 왼쪽 트리에서 **`검색`** 클릭 -> Output에 `found` 값과 찾아온 약물 정보 확인
3. 왼쪽 트리에서 **`답변생성`(또는 ChatOpenAI)** 클릭 -> Input에 **GPT가 받은 프롬프트 전문** 확인

**여기서 보이는 것**

- `검색`이 찾아온 약물 정보가, `답변생성` 프롬프트의 `[약물 정보]` 자리에 그대로 들어가 있습니다.
- RAG가 엉뚱한 답을 하면: 검색이 약물 정보를 못 가져왔으면 **검색 문제**, 잘 가져왔는데 답이 이상하면 **생성 문제**. 이 구분을 트레이스로 할 수 있습니다.


## STEP 5. 데이터셋 만들고 평가하기

추적은 "무슨 일이 있었나"를 보여줍니다. **평가**는 "그래서 잘했나"를 점수로 매깁니다.

평가에는 셋이 필요합니다: **데이터셋**(문제+정답), **타깃 함수**(평가할 앱), **평가자**(채점자).

### 5-1. 데이터셋 만들기

질문과 기대 답변을 데이터셋으로 등록합니다.


In [10]:
from langsmith import Client

client = Client()
dataset_name = "복약RAG-평가셋"

if not client.has_dataset(dataset_name=dataset_name):
    client.create_dataset(dataset_name=dataset_name,
                          description="복약상담 RAG 정확도 평가용")
    examples = [
        {"inputs": {"question": "와파린과 같이 먹으면 위험한 약은?"},
         "outputs": {"answer": "아스피린, 이부프로펜"}},
        {"inputs": {"question": "심바스타틴과 자몽주스 같이 먹어도 되나요?"},
         "outputs": {"answer": "근육손상 위험이 있어 피해야 함"}},
        {"inputs": {"question": "레보티록신은 언제 먹어야 하나요?"},
         "outputs": {"answer": "공복에 복용, 복용 후 30분 음식 피할 것"}},
        {"inputs": {"question": "암로디핀과 자몽주스 같이 먹어도 되나요?"},
         "outputs": {"answer": "혈중 농도가 상승할 수 있어 피해야 함"}},
    ]
    client.create_examples(dataset_name=dataset_name, examples=examples)
    print("데이터셋 생성:", len(examples), "문제")
else:
    print("데이터셋이 이미 있습니다:", dataset_name)


데이터셋 생성: 4 문제


### 5-2. 타깃 함수와 평가자

타깃 함수(평가할 앱)와 평가자(채점자)를 만듭니다.


In [11]:
# 타깃 함수: 데이터셋의 inputs 를 받아 답을 돌려줌
def target(inputs: dict) -> dict:
    return {"answer": rag(inputs["question"])}


# 평가자: 정답 키워드가 답변에 들어 있으면 1점, 아니면 0점
def correct_keyword(outputs: dict, reference_outputs: dict) -> dict:
    expected = reference_outputs["answer"]
    actual = outputs["answer"]
    first_word = expected.split(",")[0].strip()   # 정답의 첫 키워드
    score = 1 if first_word in actual else 0
    return {"key": "정답포함", "score": score}

print("타깃 함수 / 평가자 준비 완료")


타깃 함수 / 평가자 준비 완료


### 5-3. 평가 실행

In [12]:
from langsmith import evaluate

results = evaluate(
    target,
    data=dataset_name,
    evaluators=[correct_keyword],
    experiment_prefix="복약RAG-v1",
    metadata={"version": "1.0"},
)

print("\n평가 완료! 출력된 링크를 열어 점수 표를 확인하세요.")


View the evaluation results for experiment: '복약RAG-v1-ee699436' at:
https://smith.langchain.com/o/0e32d70b-6866-40a8-a8aa-a54aeaa25949/datasets/65b58cd9-3a6f-4ebb-8c2a-6d68b516db04/compare?selectedSessions=25dc42b7-f157-4ccb-9471-bdce025195c0




0it [00:00, ?it/s]


평가 완료! 출력된 링크를 열어 점수 표를 확인하세요.


### 대시보드에서 확인

출력된 링크를 열거나, **Datasets & Experiments -> 복약RAG-평가셋** 에서 점수 표를 봅니다.

**볼 것**: 문제별 점수, 평균 점수, Latency, Token. 틀린 문제가 있다면 앱이 진짜 틀린 건지, 표현이 달라 채점에 걸린 건지 트레이스를 열어 확인합니다.


## STEP 6. 프롬프트 고쳐서 v2 만들고 비교

마지막입니다. 프롬프트를 **개선한 v2**를 만들고, **같은 데이터셋**으로 평가해서 v1과 비교합니다.

v2에는 "관련된 약 이름을 빠짐없이 정확히 포함하라"는 지시를 추가했습니다.


In [13]:
# v2: 프롬프트 개선 (약 이름을 정확히 포함하도록 지시 추가)
@traceable(name="답변생성-v2")
def generate_v2(question: str, context: str):
    prompt = (
        "당신은 복약상담 도우미입니다. 아래 약물 정보만 근거로 한국어로 답하세요.\n"
        "관련된 약 이름을 빠짐없이 정확히 포함해서 답하세요.\n"          # ← v2 추가 지시
        "정보에 없는 내용은 지어내지 말고, 약물 복용은 의사·약사와 상의하라고 안내하세요.\n\n"
        f"[약물 정보]\n{context}\n\n[질문] {question}\n[답변]"
    )
    return llm.invoke(prompt).content


@traceable(name="복약RAG-v2")
def rag_v2(question: str):
    r = retrieve(question)
    if not r["found"]:
        return "관련 약물 정보를 찾지 못했습니다. 약 이름이나 증상을 다시 입력해 주세요."
    return generate_v2(question, r["context"])


def target_v2(inputs: dict) -> dict:
    return {"answer": rag_v2(inputs["question"])}

print("v2 준비 완료")


v2 준비 완료


v2를 같은 데이터셋·같은 평가자로 평가합니다.

In [14]:
results_v2 = evaluate(
    target_v2,
    data=dataset_name,
    evaluators=[correct_keyword],
    experiment_prefix="복약RAG-v2",
    metadata={"version": "2.0"},
)

print("\nv2 평가 완료!")


View the evaluation results for experiment: '복약RAG-v2-6aebd049' at:
https://smith.langchain.com/o/0e32d70b-6866-40a8-a8aa-a54aeaa25949/datasets/65b58cd9-3a6f-4ebb-8c2a-6d68b516db04/compare?selectedSessions=5e0c8ae8-3816-4b2c-8a19-30074a059dbf




0it [00:00, ?it/s]


v2 평가 완료!


### 대시보드에서 확인 — 비교 그래프

**Datasets & Experiments -> 복약RAG-평가셋** 으로 가서 `복약RAG-v1`과 `복약RAG-v2`를 **둘 다 체크**하고 **[Compare]** 를 누릅니다.

**볼 것 (비교 화면)**

- 상단 차트: **Feedback Scores**(점수), **Latency**(속도), **Token Count**(토큰), **Cost**(비용)를 v1 vs v2로 비교
- 아래 표: 같은 질문에 대한 v1과 v2의 **답변을 나란히** 비교

생각해볼 점: 점수는 비슷해도 v2가 약 이름을 더 정확히 포함했다면, 그 차이는 "정답포함" 평가자만으로는 잘 안 잡힙니다. 개선을 점수로 드러내려면 평가자를 그에 맞게 보강해야 합니다. 이것이 평가 설계의 핵심입니다.


---
# 정리

이 노트북에서 LangSmith의 세 가지를 모두 보았습니다.

| 기능 | 한 일 | 대시보드에서 본 것 |
| --- | --- | --- |
| 추적 | 환경변수 2줄로 켜기 | 트레이스 목록, 토큰·비용·시간 |
| @traceable | 함수에 데코레이터 | 복약RAG → 검색 → 답변생성 트리 |
| 평가 | 데이터셋 + evaluate | 점수 표 |
| 비교 | v1 vs v2 | Compare 그래프 |

- **추적**은 "무슨 일이 있었나"를 보여주고, **평가**는 "그래서 잘했나"를 점수로 증명합니다.
- 프롬프트를 고칠 때마다 v2, v3로 평가해 비교하면, "느낌상 좋아졌다"가 아니라 **숫자로** 개선을 확인할 수 있습니다.

> 제출 시: 모든 셀을 실행한 `.ipynb` 파일과 함께, LangSmith 대시보드 캡처(트레이스 트리, v1 점수 표, v1 vs v2 비교 그래프)를 첨부하세요.
